# Does the gain come from targeting or from retrying? A pre-registered control

Reviewer 7's sixth comment is the reason this file exists. Section VI.F reports that
acting on a flag raises end-to-end success from 93 to 101 of 200 tasks, and the same
section reports that the intervention arm generated 826 steps where the baseline
generated 706. Those two facts cannot be held together: an arm that is allowed more
attempts may improve for that reason alone, and the four points are therefore not
attributable to the detector. The reviewer asks for a retry baseline at a matched
computational budget, and this is its protocol.

As with the intervention experiment, **every choice is fixed here, in the repository,
before the run**. Nothing below is decided after seeing a number, and the deviation rule
at the end binds anything that changes afterwards.

## What is fixed

**Task set.** `benchmark.generate(per_template=25, seed=42)`, 200 tasks, the identical
list for every arm. This is the same call the intervention experiment made, so this
session's arm A and arm B are comparable with the published ones.

**Agent and decoding.** Qwen 2.5 1.5B-Instruct, greedy, `max_new_tokens=150`, six-action
budget, hard ceiling of ten generated steps, the tool surface and prompt of
`code/agent.py` completed as raw text. Identical for every arm.

**Arm A, baseline.** `run_agent` from the released `agent.py`, unchanged.

**Arm B, targeted intervention.** The arm of Section VI.F, reproduced here verbatim from
that notebook: `tier1` after each step, the contradiction half of `tier2` on a `finish`,
at most two interventions, retries free of the action budget, the two corrective notes
that name the kind of defect.

**Arm C1, blind retry at a matched budget.** The same loop with no detector in it. After
each step, a retry fires with probability `P_RETRY`, drawn from a generator seeded before
the run, with the same cap of two per trajectory and the same exemption from the action
budget. The note appended is `NOTE_BLIND`, which names no defect and asks for the step
again. `P_RETRY` is fixed at **161 / 706 = 0.2280**, the 161 retries arm B fired in the
published run over the 706 steps arm A generated there, so that the expected number of
retries equals arm B's. The realized count is reported rather than assumed.

**Arm C2, count-matched retry.** For each task, the number of retries arm B fired on that
task in this session, placed at positions drawn before the arm runs, uniformly over the
six action slots. A drawn position beyond the end of the trajectory would be lost, so any
unspent balance is spent on the terminating step, and nothing fires once the task's budget
is exhausted. The budget is therefore equal task by task, and spending the balance on the
terminating step mirrors arm B, whose contradiction retries fire there. This is an
analysis arm and not a deployable baseline, since knowing its own budget requires the
detector; it is reported as such.

**Why compute is matched on generated steps.** A step is one call to `model.generate`
with `max_new_tokens=150`, and generation dominates the cost of every arm. Generated
tokens are counted exactly as well, and both are reported. The blind note is shorter than
the two targeted notes, which the token count will show; the difference is in the prompt
and not in the generated length.

**Outcome.** End-to-end success decided by `code/experiments/success_checker.py`, frozen
before any of these experiments existed and asserted in the cell below to still recover
72 of 72 human successes.

**The analysis, fixed now.** The primary comparison is **arm B against arm C1**, the
paired difference in success rate at a matched budget: it is the comparison the reviewer's
question needs, and it is the one that can refute the reading of Section VI.F. Secondary
comparisons are C1 against A, C2 against C1 and C2 against A. Each one is tested by the
exact binomial on its discordant pairs and bounded by a cluster bootstrap over the eight
templates, 5,000 draws at seed 42, which is the procedure Section VI.F used and few
clusters for the reason it states there.

## What is predicted, so that it cannot be rewritten afterwards

Arm B is expected to beat arm C1. The targeted retry fires where a structural defect has
actually been observed, and the contradiction trigger fires on exactly the two templates
whose correct answer is a refusal, which is where Section VI.F's gains sit. A blind retry
spends the same budget on steps that are mostly sound.

**The refutation condition, stated before the run.** If arm C1 reaches arm B's success
count or beats it, then the four points of Section VI.F are not evidence that detection
helped, and the manuscript will say that acting on these flags is worth no more than
retrying at random on this benchmark. If the two arms are separated by a difference the
discordant pairs do not resolve, which eight clusters and 200 tasks make likely, then
nothing is established either way and the section will say that instead.

**A third outcome is possible and is also pre-registered.** Both retry arms may beat arm
A by the mechanics of retrying alone. In that case the reportable finding is that extra
attempts, not detection, carry the four points.

## What this control cannot establish

One model, one benchmark, eight clusters, one intervention design. A null result here
does not show that targeted intervention cannot help, and a positive one does not show
that it transfers. The question answered is narrow and local: on this benchmark, at this
budget, does the detector earn the extra computation it spends.

## Deviation rule

If anything in this protocol has to change once the run starts, the change is written into
this file with its reason and its date, and the result is reported as a deviation rather
than as the plan. The reproduction of arms A and B is itself a check: the published run
gives 93 and 101, and a session that does not reproduce those counts is a deviation to be
recorded, after which the comparison uses this session's arms, which are the internally
valid ones.

**One detail about the code this notebook fetches.** It loads `agent.py` and `signals.py`
from the `neucom-d-26-17269-r1` tag, that is, the versions the published intervention run
used, so that arm B here is the same arm B. Those versions carry the not-found sentinel
defect that Section VIII declares, which `experiments/sentinel_check.py` shows no reported
figure depends on, since the string carries no digit. Fetching the fixed version instead
would make arm B incomparable with the published one for no gain.

**Requires:** the `neucom-d-26-17269-r1` tag, GPU T4, internet on. Expect roughly one hour
per arm, so four hours in all; each arm writes its file as soon as it finishes, so a
session that dies is resumed by rerunning only the arms whose files are missing.

## Deviations

None yet.


In [ ]:
import ast, json, os, random, re, sys, time, types, urllib.request
import torch
from collections import Counter

TAG = ("https://raw.githubusercontent.com/achenna-source/"
       "icv-agent-observability/neucom-d-26-17269-r1/")
fetch = lambda p: urllib.request.urlopen(TAG + p, timeout=90).read().decode("utf-8")

def load_module(path, name):
    mod = types.ModuleType(name)
    mod.__dict__["__name__"] = name
    # Comme dans intervention_experiment.ipynb : les modules du depot sont executes
    # depuis une chaine, donc __file__ n'existe pas et success_checker.py s'en sert au
    # chargement pour situer data/. Le nom est fourni ici, le module n'est pas touche.
    mod.__dict__["__file__"] = "/kaggle/working/" + path
    exec(compile(fetch(path), path, "exec"), mod.__dict__)
    sys.modules[name] = mod
    return mod

agent = load_module("code/agent.py", "agent")
signals = load_module("code/signals.py", "signals")
benchmark = load_module("code/benchmark.py", "benchmark")
checker = load_module("code/experiments/success_checker.py", "success_checker")

_kb = json.loads(fetch("data/knowledge_base.json"))
KB = _kb["facts"]
POP = {k.split(" ", 1)[1]: int(v) for k, v in KB.items() if k.startswith("population ")}
CAP = {k.split(" ", 1)[1]: v for k, v in KB.items() if k.startswith("capitale ")}
KNOWN = {int(v) for v in KB.values() if str(v).isdigit()}
STORE = (POP, CAP, KNOWN)

print("modules loaded from the tag: agent, signals, benchmark, success_checker")
print("gpus: %d" % torch.cuda.device_count())


## The frozen components, verified before anything runs

The same two assertions the intervention experiment makes. If either fails, the protocol
is void and the run does not proceed.


In [ ]:
import io, tempfile
_tmp = os.path.join(tempfile.gettempdir(), "trajectories_200.jsonl")
io.open(_tmp, "w", encoding="utf-8").write(fetch("data/trajectories_200.jsonl"))
print("generator self-check : %s" % benchmark.self_check(corpus_path=_tmp))

_kbtmp = os.path.join(tempfile.gettempdir(), "knowledge_base.json")
io.open(_kbtmp, "w", encoding="utf-8").write(fetch("data/knowledge_base.json"))
checker.DATA = tempfile.gettempdir()
print("checker self-check   : %s" % checker.self_check(corpus_path=_tmp))
assert checker.self_check(corpus_path=_tmp)["recovered"] == 72, "checker drifted"

try:
    PUBLISHED = json.loads(fetch("results/intervention_result.json"))
except Exception as e:
    # Si le fichier n'est pas a ce tag, les deux comptes publies sont ceux du manuscrit,
    # Section VI.F, et la comparaison de reproduction se fait contre eux.
    print("results/intervention_result.json indisponible au tag (%s) : on prend les "
          "comptes de la Section VI.F" % type(e).__name__)
    PUBLISHED = {"A": 93, "B": 101}
assert PUBLISHED["A"] == 93 and PUBLISHED["B"] == 101, PUBLISHED
print()
print("both frozen components verify; the published run gives A %d and B %d"
      % (PUBLISHED["A"], PUBLISHED["B"]))


## The four arms, written out

Arm B is the arm of Section VI.F, copied from its notebook. Arms C1 and C2 differ from it
in two places only: no detector is consulted, and the retry decision comes from a seeded
generator. The guard below counts every call into `signals` and asserts that the blind arm
makes none.


In [ ]:
BUDGET = 6
MAX_STEPS_HARD = 10
MAX_INTERVENTIONS = 2
P_RETRY = 161.0 / 706.0          # pre-enregistre : les reprises de B sur les pas de A
SEED_RETRY = 20261106

NOTE_SCHEMA = ("Observation: Erreur de forme detectee a l'etape precedente. Verifie que "
               "le nom d'outil est l'un de knowledge_base, calculator, finish, et que "
               "l'argument est non vide. Reprends cette etape.")
NOTE_CONTRA = ("Observation: Verification: la reponse finale contient une valeur qui ne "
               "provient pas des observations obtenues. Reprends en t'appuyant "
               "uniquement sur les valeurs recuperees, ou indique que l'information "
               "est absente.")
# Le note aveugle ne nomme aucun defaut et ne dit pas qu'un defaut a ete cherche.
NOTE_BLIND = ("Observation: Reprise demandee a l'etape precedente, sans diagnostic. "
              "Reprends cette etape.")

CALLS = {"tier1": 0, "tier2": 0}
_tier1, _tier2 = signals.tier1, signals.tier2

def tier1(*a, **k):
    CALLS["tier1"] += 1
    return _tier1(*a, **k)

def tier2(*a, **k):
    CALLS["tier2"] += 1
    return _tier2(*a, **k)

def run_with_intervention(task, llm_generate, kb):
    """Arm B, verbatim from intervention_experiment.ipynb except for the counted
    wrappers around the two signals."""
    tools = {"calculator": agent.calculator,
             "knowledge_base": agent.make_knowledge_base(kb)}
    trajectory, fired, actions, total = [], [], 0, 0
    context = agent.SYSTEM_PROMPT + "\n\nTache : %s\n\n" % task
    while actions < BUDGET and total < MAX_STEPS_HARD:
        total += 1
        parsed = agent.parse_action(llm_generate(context))
        action, inp = parsed["action"], parsed["input"]
        if action == "finish":
            step = {"thought": parsed["thought"], "action": "finish",
                    "input": inp, "observation": None}
            candidate = trajectory + [step]
            _, _, contra = tier2(candidate, encoder=None)
            if contra and len(fired) < MAX_INTERVENTIONS:
                trajectory.append(step)
                fired.append(("contradiction", total))
                context += ('Thought: %s\nAction: finish\nAction Input: "%s"\n%s\n'
                            % (parsed["thought"], inp, NOTE_CONTRA))
                continue
            trajectory.append(step)
            break
        obs = (tools[action](inp) if action in tools and inp
               else "Erreur: outil inconnu '%s'" % action)
        step = {"thought": parsed["thought"], "action": action,
                "input": inp, "observation": obs}
        trajectory.append(step)
        flagged, _ = tier1(trajectory)
        if flagged and len(fired) < MAX_INTERVENTIONS:
            fired.append(("schema", total))
            context += ('Thought: %s\nAction: %s\nAction Input: "%s"\n%s\n'
                        % (parsed["thought"], action, inp, NOTE_SCHEMA))
            continue
        actions += 1
        context += ('Thought: %s\nAction: %s\nAction Input: "%s"\nObservation: %s\n'
                    % (parsed["thought"], action, inp, obs))
    return {"task": task, "trajectory": trajectory,
            "interventions": fired, "steps_generated": total}

def run_with_blind_retry(task, llm_generate, kb, decide):
    """Arms C1 and C2. `decide(step_index, n_fired, is_finish)` returns True to retry the
    step just taken. It sees the index, the number of retries already spent and whether
    the step is a terminating one, and nothing else: no detector can hide in it."""
    tools = {"calculator": agent.calculator,
             "knowledge_base": agent.make_knowledge_base(kb)}
    trajectory, fired, actions, total = [], [], 0, 0
    context = agent.SYSTEM_PROMPT + "\n\nTache : %s\n\n" % task
    while actions < BUDGET and total < MAX_STEPS_HARD:
        total += 1
        parsed = agent.parse_action(llm_generate(context))
        action, inp = parsed["action"], parsed["input"]
        if action == "finish":
            step = {"thought": parsed["thought"], "action": "finish",
                    "input": inp, "observation": None}
            if len(fired) < MAX_INTERVENTIONS and decide(total, len(fired), True):
                trajectory.append(step)
                fired.append(("blind", total))
                context += ('Thought: %s\nAction: finish\nAction Input: "%s"\n%s\n'
                            % (parsed["thought"], inp, NOTE_BLIND))
                continue
            trajectory.append(step)
            break
        obs = (tools[action](inp) if action in tools and inp
               else "Erreur: outil inconnu '%s'" % action)
        step = {"thought": parsed["thought"], "action": action,
                "input": inp, "observation": obs}
        trajectory.append(step)
        if len(fired) < MAX_INTERVENTIONS and decide(total, len(fired), False):
            fired.append(("blind", total))
            context += ('Thought: %s\nAction: %s\nAction Input: "%s"\n%s\n'
                        % (parsed["thought"], action, inp, NOTE_BLIND))
            continue
        actions += 1
        context += ('Thought: %s\nAction: %s\nAction Input: "%s"\nObservation: %s\n'
                    % (parsed["thought"], action, inp, obs))
    return {"task": task, "trajectory": trajectory,
            "interventions": fired, "steps_generated": total}

print("arms defined | retry probability %.4f | cap %d | hard ceiling %d"
      % (P_RETRY, MAX_INTERVENTIONS, MAX_STEPS_HARD))
print("blind note, %d words : %s" % (len(NOTE_BLIND.split()), NOTE_BLIND))


## The model, the task list and the cost counters

Tokens are counted here rather than reconstructed afterwards, because the cost the
reviewer asks for is the cost this session actually paid.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16,
                                             device_map="auto")
model.eval()

COST = {"calls": 0, "prompt_tokens": 0, "new_tokens": 0}

def llm_generate(prompt):
    # Deviation 2 du carnet d'intervention : completion du texte brut, sans gabarit de
    # conversation. Reprise telle quelle pour que les bras soient comparables.
    enc = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=150, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    n_prompt = enc["input_ids"].shape[1]
    COST["calls"] += 1
    COST["prompt_tokens"] += int(n_prompt)
    COST["new_tokens"] += int(out[0].shape[0] - n_prompt)
    return tok.decode(out[0][n_prompt:], skip_special_tokens=True)

TASKS = benchmark.generate(per_template=25, seed=42)
TPLS = sorted({t["template"] for t in TASKS})
IDX = {t: [i for i, x in enumerate(TASKS) if x["template"] == t] for t in TPLS}
print("%s loaded | tasks %d | templates %s"
      % (MODEL, len(TASKS), dict(Counter(t["template"] for t in TASKS))))

def finished(r):
    return any(str(s.get("action")).lower() == "finish" for s in r["trajectory"])

def run_arm(fn, tag, path):
    """Runs an arm, writes it as soon as it finishes, and records its own cost."""
    if os.path.exists(path):
        done = json.load(open(path, encoding="utf-8"))
        assert isinstance(done, dict) and "arm" in done and "meta" in done, (
            "%s n'a pas la forme attendue : un fichier d'un autre carnet porte ce nom"
            % path)
        print("   [%s] %s already on disk, %d tasks, not rerun"
              % (tag, path, len(done["arm"])))
        return done
    before = dict(COST)
    t0, out = time.time(), []
    for i, t in enumerate(TASKS):
        r = fn(i, t["task"])
        r["template"] = t["template"]
        out.append(r)
        if (i + 1) % 25 == 0:
            print("   [%s] %3d/200  %.0f s  finish so far %d"
                  % (tag, i + 1, time.time() - t0, sum(1 for x in out if finished(x))))
    meta = {"arm": tag, "seconds": round(time.time() - t0, 1),
            "calls": COST["calls"] - before["calls"],
            "prompt_tokens": COST["prompt_tokens"] - before["prompt_tokens"],
            "new_tokens": COST["new_tokens"] - before["new_tokens"],
            "generated_steps": sum(r["steps_generated"] for r in out)
            if "steps_generated" in out[0] else sum(len(r["trajectory"]) for r in out),
            "retries": sum(len(r.get("interventions", [])) for r in out),
            "reaches_finish": sum(1 for x in out if finished(x))}
    json.dump({"meta": meta, "arm": out}, open(path, "w", encoding="utf-8"),
              ensure_ascii=False)
    print("   [%s] %s" % (tag, meta))
    return {"meta": meta, "arm": out}


## The runs

Arm A and arm B first, because they are the reproduction check: the published run gives 93
and 101. Then the two retry arms. Each arm writes its own file, so a session that dies is
resumed by rerunning this notebook, which skips the arms already on disk.


In [ ]:
A = run_arm(lambda i, task: agent.run_agent(task, llm_generate, KB),
            "A baseline", "retry_arm_a.json")
CALLS["tier1"] = CALLS["tier2"] = 0
B = run_arm(lambda i, task: run_with_intervention(task, llm_generate, KB),
            "B targeted", "retry_arm_b.json")
print()
print("signals consulted during arm B : tier1 %d, tier2 %d" % (CALLS["tier1"],
                                                               CALLS["tier2"]))


In [ ]:
# Le tirage est fixe avant le bras, graine publiee dans le protocole.
_rng = random.Random(SEED_RETRY)
_draws = {i: [_rng.random() for _ in range(MAX_STEPS_HARD)] for i in range(len(TASKS))}

def decide_c1(i):
    def d(step_index, n_fired, is_finish):
        return _draws[i][step_index - 1] < P_RETRY
    return d

CALLS["tier1"] = CALLS["tier2"] = 0
C1 = run_arm(lambda i, task: run_with_blind_retry(task, llm_generate, KB, decide_c1(i)),
             "C1 blind", "retry_arm_c1.json")
assert CALLS["tier1"] == 0 and CALLS["tier2"] == 0, (
    "le bras aveugle a consulte un detecteur %s, ce qui invalide sa qualite de temoin"
    % CALLS)
print()
print("no detector was consulted during arm C1: tier1 %d, tier2 %d"
      % (CALLS["tier1"], CALLS["tier2"]))


In [ ]:
# Bras C2 : le budget de B, tache par tache, a des positions tirees au hasard.
_rng2 = random.Random(SEED_RETRY + 1)
_want = {i: len(r["interventions"]) for i, r in enumerate(B["arm"])}
_slots = {i: sorted(_rng2.sample(range(1, BUDGET + 1), _want[i])) if _want[i] else []
          for i in range(len(TASKS))}

def decide_c2(i):
    def d(step_index, n_fired, is_finish):
        # le budget de la tache est celui de B, ni plus ni moins : une position tiree
        # au-dela de la trajectoire serait perdue, donc le solde se depense sur l'etape
        # terminale, et rien ne se depense une fois le budget epuise
        if n_fired >= _want[i]:
            return False
        return step_index in _slots[i] or is_finish
    return d

CALLS["tier1"] = CALLS["tier2"] = 0
C2 = run_arm(lambda i, task: run_with_blind_retry(task, llm_generate, KB, decide_c2(i)),
             "C2 count-matched", "retry_arm_c2.json")
assert CALLS["tier1"] == 0 and CALLS["tier2"] == 0, CALLS
print()
print("retries wanted %d, retries spent %d (the balance of a task whose drawn positions "
      "fall beyond its trajectory is spent on its terminating step)"
      % (sum(_want.values()), C2["meta"]["retries"]))


## The pre-specified analysis

Nothing here chooses a statistic after seeing a number. The primary comparison is B
against C1.


In [ ]:
from math import comb

def score(a):
    return [checker.succeeded(r["task"], r["trajectory"], STORE)[0] for r in a["arm"]]

S = {"A": score(A), "B": score(B), "C1": score(C1)}
try:
    S["C2"] = score(C2)
except NameError:
    print("arm C2 absent from this session; its comparisons are skipped")

for k in sorted(S):
    print("arm %-3s success %3d/200 = %.1f%%" % (k, sum(S[k]), 100.0 * sum(S[k]) / 200))

dev = []
if sum(S["A"]) != PUBLISHED["A"] or sum(S["B"]) != PUBLISHED["B"]:
    dev.append("reproduction: this session gives A %d and B %d where the published run "
               "gives %d and %d" % (sum(S["A"]), sum(S["B"]),
                                    PUBLISHED["A"], PUBLISHED["B"]))
    print()
    print("DEVIATION TO RECORD: %s" % dev[-1])
    print("the comparisons below use this session's arms, which are the internally "
          "valid ones")

def paired(x, y):
    sx, sy = S[x], S[y]
    only_x = sum(1 for a, b in zip(sx, sy) if a and not b)
    only_y = sum(1 for a, b in zip(sx, sy) if b and not a)
    n, k = only_x + only_y, only_y
    p = (sum(comb(n, i) for i in range(min(k, n - k) + 1)) / 2 ** n * 2) if n else 1.0
    rng = random.Random(42)
    draws = []
    for _ in range(5000):
        pick = [rng.choice(TPLS) for _ in TPLS]
        sel = [i for t in pick for i in IDX[t]]
        draws.append(sum(sy[i] for i in sel) / len(sel)
                     - sum(sx[i] for i in sel) / len(sel))
    draws.sort()
    return dict(diff=round((sum(sy) - sum(sx)) / 200.0, 3),
                fixed=only_y, broken=only_x, discordant=n, p=round(min(1.0, p), 4),
                ci=[round(draws[int(0.025 * len(draws))], 3),
                    round(draws[int(0.975 * len(draws))], 3)])

PAIRS = [("C1", "B"), ("A", "C1"), ("A", "B")]
if "C2" in S:
    PAIRS += [("C1", "C2"), ("A", "C2"), ("C2", "B")]
print()
print("%-10s %8s %7s %8s %7s %8s %20s"
      % ("pair", "diff", "fixed", "broken", "disc.", "p", "IC95 by template"))
RES = {}
for x, y in PAIRS:
    r = paired(x, y)
    RES["%s vs %s" % (y, x)] = r
    print("%-10s %+8.3f %7d %8d %7d %8.4f  [%+.3f, %+.3f]"
          % ("%s - %s" % (y, x), r["diff"], r["fixed"], r["broken"],
             r["discordant"], r["p"], r["ci"][0], r["ci"][1]))

print()
print("%-18s %9s %9s %9s %9s %9s"
      % ("arm", "seconds", "steps", "retries", "new tok", "finish"))
for k, a in (("A baseline", A), ("B targeted", B), ("C1 blind", C1)) + \
            ((("C2 count-matched", C2),) if "C2" in S else ()):
    m = a["meta"]
    print("%-18s %9.0f %9d %9d %9d %9d"
          % (k, m["seconds"], m["generated_steps"], m["retries"], m["new_tokens"],
             m["reaches_finish"]))

print()
print("%-5s %6s %6s %6s %6s" % ("tmpl", "A", "B", "C1", "C2" if "C2" in S else ""))
for t in TPLS:
    row = [sum(S[k][i] for i in IDX[t]) for k in ("A", "B", "C1")]
    row += [sum(S["C2"][i] for i in IDX[t])] if "C2" in S else []
    print("%-5s %6d %6d %6d %6s" % (t, row[0], row[1], row[2],
                                    row[3] if len(row) > 3 else ""))

primary = RES["B vs C1"]
print()
print("PRIMARY, targeted against blind at a matched budget: %+.3f, %d fixed against %d "
      "broken, p = %.4f, interval [%+.3f, %+.3f]"
      % (primary["diff"], primary["fixed"], primary["broken"], primary["p"],
         primary["ci"][0], primary["ci"][1]))
if primary["ci"][0] <= 0 <= primary["ci"][1]:
    print("the interval contains zero: this control separates neither arm from the "
          "other, which is the outcome the protocol says to report as such")
elif primary["diff"] > 0:
    print("targeting is ahead of blind retry at a matched budget on this benchmark")
else:
    print("blind retry matches or beats targeting: the refutation condition of the "
          "protocol is met, and Section VI.F must say so")

json.dump({"published": PUBLISHED,
           "success": {k: sum(v) for k, v in S.items()},
           "per_template": {t: {k: sum(S[k][i] for i in IDX[t]) for k in S}
                            for t in TPLS},
           "pairs": RES,
           "cost": {k: a["meta"] for k, a in
                    (("A", A), ("B", B), ("C1", C1)) +
                    ((("C2", C2),) if "C2" in S else ())},
           "protocol": {"p_retry": round(P_RETRY, 4), "seed_retry": SEED_RETRY,
                        "cap": MAX_INTERVENTIONS, "budget": BUDGET,
                        "hard_ceiling": MAX_STEPS_HARD,
                        "note_blind": NOTE_BLIND},
           "deviations": dev},
          open("retry_baseline_result.json", "w", encoding="utf-8"), ensure_ascii=False,
          indent=1)
print()
print("written: retry_baseline_result.json")
